# ⛔ ĐƯỜNG CỤT — ĐỪNG CHẠY LẠI

Notebook này (Qwen2.5-7B few-shot trích xuất trực tiếp làm model nộp) đã chấm thật:
**11.4736** (WER 86.40 / J_assertion 14.01 / J_candidates 7.98), so với 36.32 của đường
distillation trong `train_ner_assertion_model.ipynb`. Chạy mất 1.85h GPU.

27/100 file ra rỗng do lỗi parser (đã hiểu rõ), nhưng **sửa hết cũng không cứu được**:
điểm là trung bình theo record nên 73 file tốt gánh toàn bộ 11.4736 -> trần là
`11.4736 x 100/73 ~= 15.7`, chưa bằng nửa 36.32. Trên 73 file chạy được, recall chỉ
14.7 entity/doc so với 28.8 của pipeline encoder.

Chi tiết: `worklog.md`, mục 2026-07-26 (part 2). Giữ lại chỉ để đối chiếu.


# ViettelRace — Trích xuất trực tiếp bằng Qwen2.5-7B (few-shot từ nhãn vòng 1)

Đường **submission = model self-host ≤9B** (Qwen2.5-7B, đúng luật), chạy trên Kaggle:
- Few-shot 3 ví dụ `(input→output)` từ nhãn vòng 1 curated → Qwen gán nhãn turn-2 **theo phong cách BTC**.
- Dò span chính xác; gán `candidates` bằng linker đã test (`drugs.csv`/`diagnoses.csv`/RxNorm) + mã Qwen đã validate.
- Ghi `output/{id}.json` → `/kaggle/working/output.zip`.

**Cần:** dataset `kaggle_bundle` (có `input/`, `output/`, `input_turn2/`, `scripts/`, `data/terminology/`),
**Internet: On** (tải Qwen), **GPU T4 x2**. Dùng **Save & Run All (Commit)** để tắt máy vẫn chạy.

In [ ]:
# 1) Load Qwen2.5-7B (4-bit) — model NỘP, self-host, ≤9B
import os, sys, json, re, glob, shutil
from pathlib import Path
import torch

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
try:
    import bitsandbytes  # noqa: F401
except Exception:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=False)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=_bnb, device_map="auto")
llm.eval()
print("loaded", LLM_NAME)

In [ ]:
# 2) Định vị bundle + nạp linker đã test (TerminologyMatcher/RxNormOfflineIndex)
_rp = glob.glob("/kaggle/input/**/scripts/run_pipeline.py", recursive=True)
BUNDLE = Path(_rp[0]).parents[1] if _rp else Path(".")
print("bundle:", BUNDLE)
sys.path.insert(0, str(BUNDLE / "scripts"))
from build_terminology_index import TerminologyMatcher
try:
    from build_rxnorm_rrf_index import RxNormOfflineIndex
except Exception:
    RxNormOfflineIndex = None

TERM = BUNDLE / "data" / "terminology"
drug_matcher = TerminologyMatcher(TERM / "drugs.csv")
diag_matcher = TerminologyMatcher(TERM / "diagnoses.csv")
rxidx = RxNormOfflineIndex(TERM / "rxnorm_full.csv") if (RxNormOfflineIndex and (TERM / "rxnorm_full.csv").exists()) else None

import csv as _csv
VALID_ICD = set()
if (TERM / "icd10_full.csv").exists():
    with open(TERM / "icd10_full.csv", encoding="utf-8") as _f:
        for _row in _csv.DictReader(_f):
            _c = (_row.get("code") or "").strip().upper()
            if _c:
                VALID_ICD.add(_c)
VALID_RXCUI = set()
if rxidx is not None:
    for _entries in rxidx.exact.values():
        for _rx, _ in _entries:
            VALID_RXCUI.add(_rx)
print(f"matchers: drug={len(drug_matcher.texts)} diag={len(diag_matcher.texts)} | ICD={len(VALID_ICD)} RXCUI={len(VALID_RXCUI)}")

ENTITY_TYPES = ["CHẨN_ĐOÁN", "TRIỆU_CHỨNG", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"]
ASSERTIONS = {"isNegated", "isHistorical", "isFamily"}
ASSERT_TYPES = {"CHẨN_ĐOÁN", "TRIỆU_CHỨNG", "THUỐC"}
CAND_TYPES = {"CHẨN_ĐOÁN", "THUỐC"}
VALID_TYPES = set(ENTITY_TYPES)
ICD_RE = re.compile(r"^[A-Z]\d{2}(?:\.\d{1,4})?$")
RX_RE = re.compile(r"^\d{1,8}$")

In [ ]:
# 3) Few-shot từ nhãn vòng 1 (dạy Qwen phong cách gán nhãn của BTC)
FEWSHOT_IDS = ["15", "62", "99"]
SYS = (
    "Bạn là chuyên gia bóc tách thực thể y khoa từ văn bản lâm sàng tiếng Việt (cuộc thi ViettelRace).\n"
    "Bóc tách MỌI thực thể, 5 loại: CHẨN_ĐOÁN, TRIỆU_CHỨNG, THUỐC, TÊN_XÉT_NGHIỆM, KẾT_QUẢ_XÉT_NGHIỆM.\n"
    "assertions (chỉ CHẨN_ĐOÁN/TRIỆU_CHỨNG/THUỐC): isNegated, isHistorical, isFamily; [] nếu không có.\n"
    "candidates: CHẨN_ĐOÁN -> mã ICD-10; THUỐC -> mã RxNorm theo liều+dạng; loại khác không có candidates.\n"
    "'text' PHẢI là chuỗi con XUẤT HIỆN CHÍNH XÁC trong văn bản. Học theo phong cách gán nhãn của các ví dụ "
    "mẫu bên dưới (ranh giới span, cách chọn loại, độ chi tiết).\n"
    'CHỈ trả về JSON array [{"text":...,"type":...,"assertions":[...],"candidates":[...]}], KHÔNG giải thích.'
)

def _load_pair(fid):
    _it = next(iter(glob.glob(f"/kaggle/input/**/input/{fid}.txt", recursive=True)), None)
    _oj = next(iter(glob.glob(f"/kaggle/input/**/output/{fid}.json", recursive=True)), None)
    if not _it or not _oj:
        return None
    _text = Path(_it).read_text(encoding="utf-8")
    _clean = []
    for _e in json.load(open(_oj, encoding="utf-8")):
        _o = {"text": _e["text"], "type": _e["type"]}
        if _e["type"] in ASSERT_TYPES:
            _o["assertions"] = _e.get("assertions", [])
        if _e["type"] in CAND_TYPES:
            _o["candidates"] = _e.get("candidates", [])
        _clean.append(_o)
    return _text, json.dumps(_clean, ensure_ascii=False)

FEWSHOT = [pr for fid in FEWSHOT_IDS if (pr := _load_pair(fid))]
print(f"few-shot examples: {len(FEWSHOT)}")

In [ ]:
# 4) Hàm trích xuất (Qwen few-shot) + dò span + gán candidate
@torch.no_grad()
def extract(text):
    msgs = [{"role": "system", "content": SYS}]
    for _in, _out in FEWSHOT:
        msgs.append({"role": "user", "content": _in})
        msgs.append({"role": "assistant", "content": _out})
    msgs.append({"role": "user", "content": text})
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**ids, max_new_tokens=1536, do_sample=False, pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\[.*\]", gen, re.S)
    if not m:
        return []
    try:
        arr = json.loads(m.group(0))
    except Exception:
        return []
    return arr if isinstance(arr, list) else []

def locate(raw, items):
    used = set(); out = []; low = raw.lower()
    for it in (items if isinstance(items, list) else []):
        if not isinstance(it, dict):
            continue
        t = str(it.get("text", "")).strip(); typ = str(it.get("type", "")).strip()
        if not t or typ not in VALID_TYPES:
            continue
        a = [x for x in (it.get("assertions") or []) if x in ASSERTIONS]
        if typ not in ASSERT_TYPES:
            a = []
        raw_c = [str(c).strip() for c in (it.get("candidates") or []) if str(c).strip()]
        span = None; s = 0
        while True:
            idx = raw.find(t, s)
            if idx < 0:
                break
            sp = (idx, idx + len(t))
            if sp not in used:
                span = sp; break
            s = idx + 1
        if span is None:
            s = 0; lt = t.lower()
            while True:
                idx = low.find(lt, s)
                if idx < 0:
                    break
                sp = (idx, idx + len(lt))
                if sp not in used:
                    span = sp; break
                s = idx + 1
        if span is None:
            pat = re.compile(r"\s+".join(re.escape(w) for w in t.split()), re.I)
            for mm in pat.finditer(raw):
                if (mm.start(), mm.end()) not in used:
                    span = (mm.start(), mm.end()); break
        if span is None:
            continue
        used.add(span)
        item = {"text": raw[span[0]:span[1]], "type": typ, "position": [span[0], span[1]]}
        if typ in ASSERT_TYPES:
            item["assertions"] = sorted(set(a))
        item["_c"] = raw_c
        out.append(item)
    out.sort(key=lambda e: (e["position"][0], e["position"][1]))
    return out

def resolve(ent):
    raw_c = ent.pop("_c", [])
    typ = ent["type"]
    if typ not in CAND_TYPES:
        return ent
    txt = ent["text"]; c = []
    if typ == "CHẨN_ĐOÁN":
        c = list(diag_matcher.lookup(txt))
        if not c:
            for x in raw_c:
                xx = x.upper().replace(" ", "")
                if ICD_RE.match(xx) and (not VALID_ICD or xx in VALID_ICD):
                    c.append(xx)
    else:
        c = list(drug_matcher.lookup(txt))
        if not c and rxidx is not None:
            c = list(rxidx.lookup(txt))
        if not c:
            for x in raw_c:
                if RX_RE.match(x) and (not VALID_RXCUI or x in VALID_RXCUI):
                    c.append(x)
    ent["candidates"] = sorted(set(c))
    return ent

In [ ]:
# 5) Chạy trên 100 file input_turn2 -> output/{id}.json -> output.zip
_tg = sorted({Path(p) for p in glob.glob("/kaggle/input/**/input_turn2/*.txt", recursive=True)
              if Path(p).stem.isdigit()}, key=lambda p: int(p.stem))
if not _tg and Path("input_turn2").exists():
    _tg = sorted(Path("input_turn2").glob("*.txt"), key=lambda p: int(p.stem))
assert _tg, "không thấy input_turn2/*.txt trong dataset"
print(f"extract {len(_tg)} docs")

OUT = Path("/kaggle/working/output")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
n_ent = 0; bad = 0
for i, p in enumerate(_tg, 1):
    raw = p.read_text(encoding="utf-8")
    try:
        ents = locate(raw, extract(raw))
    except Exception as e:
        print("  [warn]", p.stem, repr(e)); ents = []
    ents = [resolve(e) for e in ents]
    for e in ents:
        if raw[e["position"][0]:e["position"][1]] != e["text"]:
            bad += 1
    (OUT / f"{p.stem}.json").write_text(json.dumps(ents, ensure_ascii=False, indent=1), encoding="utf-8")
    n_ent += len(ents)
    if i % 10 == 0:
        print(f"  {i}/{len(_tg)} (entities {n_ent})")

_zp = Path("/kaggle/working/output")
if _zp.with_suffix(".zip").exists():
    _zp.with_suffix(".zip").unlink()
shutil.make_archive(str(_zp), "zip", root_dir="/kaggle/working", base_dir="output")
print(f"WROTE {len(_tg)} files, {n_ent} entities, span-bad={bad}")
print("=> /kaggle/working/output.zip  (tải từ tab Output)")